In [431]:
import pandas as pd
import numpy as np
from pathlib import Path

In [432]:
from pathlib import Path

BASE_DIR = Path("..")

INPUT_FILE = (
    BASE_DIR
    / "data"
    / "processed"
    / "DataSetLimpio_V2.csv"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "dimensional"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [433]:
df = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig"
)

print(f"Registros cargados: {len(df):,}")
print(df.shape)

Registros cargados: 805,243
(805243, 8)


In [434]:
df.columns.tolist()

['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'CustomerID',
 'Country']

In [435]:
required_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "CustomerID",
    "Country"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Columnas faltantes:")
    print(missing_columns)
else:
    print("Todas las columnas requeridas están presentes.")

Todas las columnas requeridas están presentes.


In [436]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 805243 entries, 0 to 805242
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      805243 non-null  int64  
 1   StockCode    805243 non-null  str    
 2   Description  805243 non-null  str    
 3   Quantity     805243 non-null  int64  
 4   InvoiceDate  805243 non-null  str    
 5   Price        805243 non-null  float64
 6   CustomerID   805243 non-null  float64
 7   Country      805243 non-null  str    
dtypes: float64(2), int64(2), str(4)
memory usage: 49.1 MB


In [437]:
df["Invoice"] = df["Invoice"].astype(str)

df["StockCode"] = df["StockCode"].astype(str)

df["CustomerID"] = (
    df["CustomerID"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

df["Country"] = df["Country"].astype(str)

df["Description"] = df["Description"].astype(str)

In [438]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

In [439]:
print(df["InvoiceDate"].min())
print(df["InvoiceDate"].max())

2009-12-01 07:45:00
2011-12-09 12:50:00


In [440]:
df["Total"] = (
    df["Quantity"] * df["Price"]
)

In [441]:
df[
    [
        "Quantity",
        "Price",
        "Total"
    ]
].head()

,Quantity,Price,Total
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


## Crear dimensión de productos

In [442]:
# Crear dimensión de productos
dim_product = (
    df[
        [
            "StockCode",
            "Description"
        ]
    ]
    .dropna(subset=["StockCode"])
    .sort_values(["StockCode", "Description"])
    .drop_duplicates(subset=["StockCode"], keep="first")
    .reset_index(drop=True)
)

dim_product.insert(
    0,
    "ProductKey",
    range(1, len(dim_product) + 1)
)

dim_product.head()

,ProductKey,StockCode,Description
0,1,10002,INFLATABLE POLITICAL GLOBE
1,2,10080,GROOVY CACTUS INFLATABLE
2,3,10109,BENDY COLOUR PENCILS
3,4,10120,DOGGY RUBBER
4,5,10123C,HEARTS WRAPPING TAPE


In [443]:
df = df.merge(
    dim_product[
        [
            "ProductKey",
            "StockCode"
        ]
    ],
    on="StockCode",
    how="left"
)

In [444]:
print(
    f"Productos en dimensión: "
    f"{len(dim_product):,}"
)

Productos en dimensión: 4,630


In [445]:
dim_product.info()

<class 'pandas.DataFrame'>
RangeIndex: 4630 entries, 0 to 4629
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ProductKey   4630 non-null   int64
 1   StockCode    4630 non-null   str  
 2   Description  4630 non-null   str  
dtypes: int64(1), str(2)
memory usage: 108.6 KB


In [446]:
dim_product.isnull().sum()

ProductKey     0
StockCode      0
Description    0
dtype: int64

In [447]:
dim_product["StockCode"].duplicated().sum()

np.int64(0)

In [448]:
print("Filas:", len(dim_product))
print("StockCode únicos:", dim_product["StockCode"].nunique())
print(
    "Duplicados:",
    dim_product["StockCode"].duplicated().sum()
)

Filas: 4630
StockCode únicos: 4630
Duplicados: 0


In [449]:
dim_product[
    dim_product["StockCode"].duplicated(keep=False)
].sort_values("StockCode").head(20)

,ProductKey,StockCode,Description


## Crear dimensión de clientes

In [450]:
dim_customer = (
    df[
        [
            "CustomerID"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [451]:
dim_customer.insert(
    0,
    "CustomerKey",
    range(
        1,
        len(dim_customer) + 1
    )
)

In [452]:
print(
    f"Clientes en dimensión: "
    f"{len(dim_customer):,}"
)

Clientes en dimensión: 5,868


## Crear dimensión de países

In [453]:
dim_country = (
    df[
        [
            "Country"
        ]
    ]
    .drop_duplicates()
    .sort_values("Country")
    .reset_index(drop=True)
)

In [454]:
dim_country.insert(
    0,
    "CountryKey",
    range(
        1,
        len(dim_country) + 1
    )
)

In [455]:
dim_country.head(10)

,CountryKey,Country
0,1,Australia
1,2,Austria
2,3,Bahrain
3,4,Belgium
4,5,Brazil
5,6,Canada
6,7,Channel Islands
7,8,Cyprus
8,9,Czech Republic
9,10,Denmark


In [456]:
print(
    f"Países en dimensión: "
    f"{len(dim_country):,}"
)

Países en dimensión: 41


## Crear dimensión de fechas

In [457]:
dates = (
    df["InvoiceDate"]
    .dt.normalize()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

In [458]:
dim_date = pd.DataFrame({
    "Date": dates
})

In [459]:
dim_date["DateKey"] = (
    dim_date["Date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["Year"] = (
    dim_date["Date"]
    .dt.year
)

dim_date["Month"] = (
    dim_date["Date"]
    .dt.month
)

dim_date["MonthName"] = (
    dim_date["Date"]
    .dt.month_name()
)

dim_date["Quarter"] = (
    "Q" +
    dim_date["Date"].dt.quarter.astype(str)
)

dim_date["Day"] = (
    dim_date["Date"]
    .dt.day
)

dim_date["DayOfWeekNumber"] = (
    dim_date["Date"]
    .dt.dayofweek + 1
)

dim_date["DayOfWeek"] = (
    dim_date["Date"]
    .dt.day_name()
)

In [460]:
dim_date = dim_date[
    [
        "DateKey",
        "Date",
        "Year",
        "Month",
        "MonthName",
        "Quarter",
        "Day",
        "DayOfWeekNumber",
        "DayOfWeek"
    ]
]

## Crear dimensión de tiempo

In [461]:
hours = sorted(
    df["InvoiceDate"]
    .dt.hour
    .dropna()
    .unique()
)

In [462]:
dim_time = pd.DataFrame({
    "Hour": hours
})

In [463]:
dim_time["TimeKey"] = (
    dim_time["Hour"]
    .astype(int)
)

In [464]:
def classify_part_of_day(hour):

    if hour < 12:
        return "Mañana"

    elif hour < 18:
        return "Tarde"

    else:
        return "Noche"

In [465]:
dim_time["PartOfDay"] = (
    dim_time["Hour"]
    .apply(classify_part_of_day)
)

In [466]:
dim_time = dim_time[
    [
        "TimeKey",
        "Hour",
        "PartOfDay"
    ]
]

dim_time.head()

,TimeKey,Hour,PartOfDay
0,6,6,Mañana
1,7,7,Mañana
2,8,8,Mañana
3,9,9,Mañana
4,10,10,Mañana


In [467]:
dim_time = dim_time[
    [
        "TimeKey",
        "Hour",
        "PartOfDay"
    ]
]

## Crear las claves en la tabla de hechos

In [468]:
df = df.merge(
    dim_customer[
        [
            "CustomerKey",
            "CustomerID"
        ]
    ],
    on="CustomerID",
    how="left"
)

In [469]:
df = df.merge(
    dim_country[
        [
            "CountryKey",
            "Country"
        ]
    ],
    on="Country",
    how="left"
)

## Crear DateKey

In [470]:
df["DateKey"] = (
    df["InvoiceDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

## Crear TimeKey

In [471]:
df["TimeKey"] = (
    df["InvoiceDate"]
    .dt.hour
    .astype(int)
)

In [472]:
print(df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country', 'Total', 'ProductKey', 'CustomerKey', 'CountryKey', 'DateKey', 'TimeKey']


## Crear la tabla de hechos

In [473]:
fact_sales = df[
    [
        "Invoice",
        "ProductKey",
        "CustomerKey",
        "CountryKey",
        "DateKey",
        "TimeKey",
        "Quantity",
        "Price",
        "Total",
        "InvoiceDate"
    ]
].copy()

In [474]:
fact_sales["Total"] = (
    fact_sales["Quantity"] * fact_sales["Price"]
)

print(fact_sales["Total"].sum())

17081814.578


## Revisar la tabla de hechos

In [475]:
fact_sales.head()

,Invoice,ProductKey,CustomerKey,CountryKey,DateKey,TimeKey,Quantity,Price,Total,InvoiceDate
0,489434,4009,1,39,20091201,7,12,6.95,83.4,2009-12-01 07:45:00
1,489434,3326,1,39,20091201,7,12,6.75,81.0,2009-12-01 07:45:00
2,489434,3328,1,39,20091201,7,12,6.75,81.0,2009-12-01 07:45:00
3,489434,1254,1,39,20091201,7,48,2.10,100.8,2009-12-01 07:45:00
4,489434,619,1,39,20091201,7,24,1.25,30.0,2009-12-01 07:45:00


In [476]:
fact_sales.shape

(805243, 10)

In [477]:
print(
    "Dataset limpio:",
    f"{len(df):,}"
)

print(
    "Tabla fact_sales:",
    f"{len(fact_sales):,}"
)

assert len(fact_sales) == len(df)

print(
    "✓ Validación exitosa: "
    "no se perdieron registros."
)

Dataset limpio: 805,243
Tabla fact_sales: 805,243
✓ Validación exitosa: no se perdieron registros.


## Validar las claves foráneas

In [478]:
print(
    "ProductKey nulos:",
    fact_sales["ProductKey"].isna().sum()
)

print(
    "CustomerKey nulos:",
    fact_sales["CustomerKey"].isna().sum()
)

print(
    "CountryKey nulos:",
    fact_sales["CountryKey"].isna().sum()
)

print(
    "DateKey nulos:",
    fact_sales["DateKey"].isna().sum()
)

print(
    "TimeKey nulos:",
    fact_sales["TimeKey"].isna().sum()
)

ProductKey nulos: 0
CustomerKey nulos: 0
CountryKey nulos: 0
DateKey nulos: 0
TimeKey nulos: 0


In [479]:
print("========== PRICE ==========")

print("Tipo de dato:")
print(fact_sales["Price"].dtype)

print("\nEstadísticos:")
print(fact_sales["Price"].describe())

print("\nValores máximos:")
print(
    fact_sales["Price"]
    .sort_values(ascending=False)
    .head(20)
)

========== PRICE ==========
Tipo de dato:
float64

Estadísticos:
count    805243.000000
mean          2.986058
std           4.108475
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max         243.630000
Name: Price, dtype: float64

Valores máximos:
48225     243.63
184071    241.38
100458    241.38
801785    240.00
679923    240.00
504485    239.30
281264    237.85
136800    230.00
268088    229.46
516564    222.75
150614    220.00
312442    220.00
795275    219.50
15322     216.00
277464    208.63
614948    208.34
48160     205.82
48159     201.56
78637     200.00
150972    200.00
Name: Price, dtype: float64


In [480]:
print("Precio mínimo:", fact_sales["Price"].min())
print("Precio máximo:", fact_sales["Price"].max())
print("Precio promedio:", fact_sales["Price"].mean())
print("Precio mediana:", fact_sales["Price"].median())

Precio mínimo: 0.001
Precio máximo: 243.63
Precio promedio: 2.9860577341249788
Precio mediana: 1.95


In [481]:
df_check = pd.read_csv(
    r"C:\Proyectos_soy_henry\Proyecto_Final\Sistema_de_reomendacion\data\dimensional\fact_sales.csv",
    sep=";"
)

df_check.head()

,Invoice,ProductKey,CustomerKey,CountryKey,DateKey,TimeKey,Quantity,Price,Total,InvoiceDate,Total_Calculado
0,489434,4009,1,39,20091201,7,12,"6,95","83,4",2009-12-01 07:45:00,"83,4"
1,489434,3326,1,39,20091201,7,12,"6,75","81,0",2009-12-01 07:45:00,"81,0"
2,489434,3328,1,39,20091201,7,12,"6,75","81,0",2009-12-01 07:45:00,"81,0"
3,489434,1254,1,39,20091201,7,48,"2,1","100,80000000000001",2009-12-01 07:45:00,"100,80000000000001"
4,489434,619,1,39,20091201,7,24,"1,25","30,0",2009-12-01 07:45:00,"30,0"


In [482]:
df_check["Price"].describe()

count     805243
unique       549
top         1,25
freq       97986
Name: Price, dtype: object

In [483]:
print("Total de ventas:")
print(fact_sales["Total"].sum())

print("\nEstadísticos de Total:")
print(fact_sales["Total"].describe())

Total de ventas:
17081814.578

Estadísticos de Total:
count    805243.000000
mean         21.213242
std          62.481157
min           0.001000
25%           4.950000
50%          11.850000
75%          19.500000
max        7144.720000
Name: Total, dtype: float64


In [484]:
fact_sales["Total_Calculado"] = (
    fact_sales["Quantity"] * fact_sales["Price"]
)

print("Total original:")
print(fact_sales["Total"].sum())

print("\nTotal calculado:")
print(fact_sales["Total_Calculado"].sum())

print("\nDiferencia:")
print(
    fact_sales["Total"].sum()
    - fact_sales["Total_Calculado"].sum()
)

Total original:
17081814.578

Total calculado:
17081814.578

Diferencia:
0.0


In [485]:
print(
    (
        fact_sales["Total"].round(2)
        != fact_sales["Total_Calculado"].round(2)
    ).sum()
)

0


## Exportar las tablas a CSV

In [486]:
fact_sales.to_csv(
    OUTPUT_DIR / "fact_sales.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

dim_product.to_csv(
    OUTPUT_DIR / "dim_product.csv",
    index=False,
    encoding="utf-8-sig",
    sep = ";",
    decimal = ","
)

dim_customer.to_csv(
    OUTPUT_DIR / "dim_customer.csv",
    index=False,
    encoding="utf-8-sig",
    sep = ";",
    decimal = ","
)

dim_country.to_csv(
    OUTPUT_DIR / "dim_country.csv",
    index=False,
    encoding="utf-8-sig",
    sep = ";",
    decimal = ","
)

dim_date.to_csv(
    OUTPUT_DIR / "dim_date.csv",
    index=False,
    encoding="utf-8-sig",
    sep = ";",
    decimal = ","
)

dim_time.to_csv(
    OUTPUT_DIR / "dim_time.csv",
    index=False,
    encoding="utf-8-sig",
    sep = ";",
    decimal = ","
)

In [487]:
for file in OUTPUT_DIR.glob("*.csv"):
    print(file.name)

dim_country.csv
dim_customer.csv
dim_date.csv
dim_product.csv
dim_time.csv
fact_sales.csv
